# GLASS-JWST NIRSpec Spectral Viewer — Abell 2744

Two viewing modes controlled by `PLOT_MODE` in the Parameters cell:

| Mode | Description |
|---|---|
| `"range"` | Plot a range of spectra (IDX_START–IDX_END), one panel per target, single redshift |
| `"z_quad"` | Plot one **2×2 figure per target** showing the same spectrum at four z-windows simultaneously — useful for visual redshift identification |

**Each `_spec.fits` (produced by msaexp) contains:**
- EXT 0 — Primary HDU (metadata only)
- EXT 1 — 2-D spectral trace image
- EXT 2+ — 1-D extracted spectrum (BINTABLE with WAVE / FLUX / ERR columns)

---


In [ ]:
# ============================================================
#  PARAMETERS  — edit these, then Kernel → Restart & Run All
# ============================================================

# Path to the root of your mastDownload tree
MAST_ROOT = "./mastDownload/HLSP"

# ── Plot mode ────────────────────────────────────────────────
# "range"  → one panel per spectrum, single redshift (original behaviour)
# "z_quad" → one 2×2 figure per spectrum, four z-windows per target
PLOT_MODE = "z_quad"

# Index range of spectra to plot (inclusive, zero-based)
IDX_START = 0
IDX_END   = 9

# ── z_quad mode: four redshift windows ───────────────────────
# Each entry is (label, z_value).  Edit z values to taste.
Z_WINDOWS = [
    ("Cluster  z ~ 0.31",  0.308),   # Abell 2744 cluster redshift
    ("Low-z   z ~ 1.5",   1.5),      # MgII 2798, [OII] 3727 in PRISM window
    ("Mid-z   z ~ 4.0",   4.0),      # Lyα, CIII], CIV enter NIR grating range
    ("High-z  z ~ 7.0",   7.0),      # High-z GLASS-JWST science targets
]

# ── range mode: single redshift override ─────────────────────
# Set to None to use per-file header value instead.
Z_OVERRIDE = None

# Set True to show the 2-D spectral trace above each 1-D panel (range mode only)
SHOW_2D = False

# ── Emission line annotation ─────────────────────────────────
SHOW_LINES = True

ANNOTATE_GROUPS = {
    "hydrogen",
    "forbidden",
    "agn_uv",
    "agn_coronal",
    "sfr",
}

# Set True to save each figure as a PNG
SAVE_PNG = True

# Output directory for saved PNGs
OUTPUT_DIR = "."


## 1 · Imports & display setup

In [ ]:
import glob
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import AsinhNorm
from astropy.io import fits
from astropy.wcs import FITSFixedWarning
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FITSFixedWarning)
warnings.filterwarnings("ignore", category=fits.verify.VerifyWarning)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

_MAST_ROOT  = Path(MAST_ROOT)
_OUTPUT_DIR = Path(OUTPUT_DIR)

_WAVE_NAMES = ["WAVE", "WAVELENGTH", "LAMBDA"]
_FLUX_NAMES = ["FLUX", "FLUX_CORR", "FNU"]
_ERR_NAMES  = ["ERR",  "UNCERTAINTY", "FLUX_ERR", "ERROR"]

COLS_PER_ROW = 2
FIG_WIDTH    = 16
PANEL_HEIGHT = 3.8

print("Imports OK ✓")


## 2 · Emission line catalogue

Rest-frame wavelengths (Å) for lines routinely targeted by JWST NIRSpec
galaxy and AGN programmes. Lines are grouped into five physical categories
and colour-coded in the plots.

| Group | Colour | Lines |
|---|---|---|
| `hydrogen` | tomato | Lyα, Hα, Hβ, Hγ, Hδ, Paα, Paβ, Paγ, Paδ, Brα, Brβ, Brγ |
| `forbidden` | gold | [OII], [OIII], [NII], [SII], [SIII], [NeIII], [ArIII] |
| `agn_uv` | orchid | NV, CIV, HeII, CIII], MgII, [NeIV], [NeV] |
| `agn_coronal` | deepskyblue | [FeVII], [FeX], [FeXI], [FeXIV] |
| `sfr` | limegreen | classic SFR / BPT diagnostics (Hα, Hβ, [OII]3727, [OIII]5007) |

> Lines are shifted to the **observed** frame using the redshift extracted from
> the FITS header (`REDSHIFT`, `Z_SPEC`, or `Z_PHOT` in that priority order).
> If no redshift is available the annotation is skipped for that spectrum.


In [ ]:
# ---------------------------------------------------------------------------
#  Emission line catalogue
#  (rest-frame wavelengths in Ångströms)
# ---------------------------------------------------------------------------
#
#  References used to compile this list:
#   • Mascia et al. 2024 (GLASS-JWST spectroscopic release)
#   • Tang et al. 2025 (JWST NIRSpec high-ionisation lines)
#   • Osterbrock & Ferland 2006 (Astrophysics of Gaseous Nebulae)
#   • Véron-Cetty & Véron 2006 AGN line atlas
#   • NIST Atomic Spectra Database
# ---------------------------------------------------------------------------

EMISSION_LINES = {

    # ── Hydrogen recombination series ──────────────────────────────────────
    "hydrogen": {
        "color"  : "tomato",
        "lw"     : 0.9,
        "alpha"  : 0.85,
        "lines"  : {
            # Lyman series (UV)
            "Lyα"   : 1215.67,
            "Lyβ"   : 1025.72,
            "Lyγ"   : 972.54,
            # Balmer series (optical)
            "Hα"    : 6562.80,
            "Hβ"    : 4861.33,
            "Hγ"    : 4340.47,
            "Hδ"    : 4101.74,
            "Hε"    : 3970.07,
            "H8"    : 3889.05,
            "H9"    : 3835.39,
            # Paschen series (NIR)
            "Paα"   : 18751.0,
            "Paβ"   : 12818.1,
            "Paγ"   : 10938.1,
            "Paδ"   : 10049.4,
            "Paε"   : 9546.0,
            # Brackett series (NIR)
            "Brα"   : 40522.0,
            "Brβ"   : 26258.0,
            "Brγ"   : 21661.0,
            "Brδ"   : 19446.0,
        },
    },

    # ── Forbidden / collisionally excited lines ────────────────────────────
    "forbidden": {
        "color"  : "gold",
        "lw"     : 0.8,
        "alpha"  : 0.80,
        "lines"  : {
            # Oxygen
            "[OII]3726"  :  3726.03,
            "[OII]3729"  :  3728.82,
            "[OIII]4363" :  4363.21,
            "[OIII]4959" :  4958.92,
            "[OIII]5007" :  5006.84,
            "[OI]6300"   :  6300.30,
            "[OI]6364"   :  6363.78,
            # Nitrogen
            "[NII]5755"  :  5754.64,
            "[NII]6548"  :  6548.05,
            "[NII]6583"  :  6583.45,
            # Sulphur
            "[SII]6716"  :  6716.44,
            "[SII]6731"  :  6730.82,
            "[SIII]9069" :  9068.60,
            "[SIII]9532" :  9531.10,
            # Neon
            "[NeIII]3869":  3868.76,
            "[NeIII]3967":  3967.47,
            "[NeV]3346"  :  3345.83,
            "[NeV]3426"  :  3425.88,
            # Argon
            "[ArIII]7135":  7135.78,
            "[ArIII]7751":  7751.10,
            "[ArIV]4711" :  4711.37,
            "[ArIV]4740" :  4740.17,
            # Helium
            "HeI 5876"   :  5875.67,
            "HeI 6678"   :  6678.15,
            "HeI 7065"   :  7065.22,
            "HeI 10830"  : 10830.34,
        },
    },

    # ── AGN / high-ionisation UV lines ─────────────────────────────────────
    # Seen in Type I/II AGN and "Little Red Dots" with JWST NIRSpec.
    # Key refs: Tang+2025, Scholtz+2023, Mazzolari+2024
    "agn_uv": {
        "color"  : "orchid",
        "lw"     : 1.0,
        "alpha"  : 0.90,
        "lines"  : {
            # UV metal lines
            "NV 1238"    :  1238.82,   # doublet; AGN ionisation tracer
            "NV 1242"    :  1242.80,
            "CIV 1548"   :  1548.20,   # doublet; broad in Type I AGN
            "CIV 1551"   :  1550.78,
            "HeII 1640"  :  1640.42,   # high-ionisation; AGN / Pop III
            "OIII] 1661" :  1660.81,   # semi-forbidden doublet
            "OIII] 1666" :  1666.15,
            "CIII] 1907" :  1906.68,   # semi-forbidden doublet; UV SFR/AGN
            "CIII] 1909" :  1908.73,
            "CII] 2326"  :  2326.11,
            "MgII 2796"  :  2795.53,   # doublet; broad in Type I AGN
            "MgII 2803"  :  2802.71,
            "[NeIV] 2423":  2422.56,   # doublet; AGN narrow line region
            "[NeIV] 2426":  2425.14,
            "[NeV] 3346" :  3345.83,   # high-ionisation; AGN
            "[NeV] 3426" :  3425.88,
            "HeII 4686"  :  4685.68,   # optical HeII; AGN / WR stars
            "[FeII] 1.26μ": 12570.0,   # NIR FeII; AGN / shock tracer
            "[FeII] 1.64μ": 16440.0,
        },
    },

    # ── AGN coronal lines ──────────────────────────────────────────────────
    # Arise in the highly ionised coronal line region close to the AGN.
    # Detectable with JWST NIRSpec in the rest-frame optical/NIR.
    # Refs: Gilli+2010, Murayama+1998, Riffel+2021
    "agn_coronal": {
        "color"  : "deepskyblue",
        "lw"     : 0.8,
        "alpha"  : 0.80,
        "lines"  : {
            "[FeVII] 3586" :  3586.32,
            "[FeVII] 5159" :  5158.89,
            "[FeVII] 5721" :  5720.70,
            "[FeVII] 6087" :  6087.00,
            "[FeX]  6374"  :  6374.51,
            "[FeXI] 7892"  :  7891.80,
            "[FeXIV] 5303" :  5302.86,
            "[SiVI] 1.963μ": 19630.0,
            "[CaVIII] 2.32μ": 23210.0,
            "[SiVII] 2.48μ": 24820.0,
            "[MgVIII] 3.03μ": 30280.0,
            "[MgVII] 5.50μ": 55030.0,
        },
    },

    # ── Classic SFR / BPT diagnostic lines ────────────────────────────────
    # This group is a curated subset of the above for users who only want
    # the handful of lines used in BPT diagrams and SFR calibrations.
    # Displayed in a distinct bright green to stand out.
    "sfr": {
        "color"  : "limegreen",
        "lw"     : 1.2,
        "alpha"  : 0.95,
        "lines"  : {
            "[OII] 3727"  :  3727.43,  # blended doublet centroid
            "Hβ"          :  4861.33,
            "[OIII] 4959" :  4958.92,
            "[OIII] 5007" :  5006.84,
            "Hα"          :  6562.80,
            "[NII] 6583"  :  6583.45,
            "[SII] 6716"  :  6716.44,
            "[SII] 6731"  :  6730.82,
        },
    },
}


# ---------------------------------------------------------------------------
#  Annotation helper
# ---------------------------------------------------------------------------

def annotate_emission_lines(
    ax,
    wave_obs: np.ndarray,
    ylo: float,
    yhi: float,
    redshift: float | None,
    groups: set[str],
) -> None:
    """
    Overplot vertical lines + labels for emission lines that fall within
    the observed wavelength window of this spectrum.

    Parameters
    ----------
    ax        : matplotlib Axes (the 1-D spectrum panel)
    wave_obs  : observed wavelength array in Å
    ylo, yhi  : current y-axis limits (used to position labels)
    redshift  : source redshift; if None the annotation is skipped
    groups    : set of group keys from EMISSION_LINES to draw
    """
    if redshift is None:
        return

    z    = float(redshift)
    wmin = wave_obs.min()
    wmax = wave_obs.max()
    span = yhi - ylo

    # Track x-positions already labelled to stagger overlapping labels
    labelled_x: list[float] = []

    for grp_key in groups:
        if grp_key not in EMISSION_LINES:
            continue
        grp     = EMISSION_LINES[grp_key]
        color   = grp["color"]
        lw      = grp["lw"]
        alpha   = grp["alpha"]

        for name, lam_rest in grp["lines"].items():
            lam_obs = lam_rest * (1.0 + z)
            if not (wmin <= lam_obs <= wmax):
                continue

            # Vertical line
            ax.axvline(lam_obs, color=color, linewidth=lw,
                       alpha=alpha, linestyle="--", zorder=4)

            # Stagger label height to reduce collision
            n_nearby = sum(1 for x in labelled_x if abs(x - lam_obs) < 180)
            y_frac   = 0.97 - 0.12 * (n_nearby % 5)
            y_pos    = ylo + y_frac * span

            ax.text(
                lam_obs, y_pos,
                name,
                color=color,
                fontsize=8,
                rotation=90,
                va="top",
                ha="center",
                alpha=min(alpha + 0.1, 1.0),
                zorder=5,
                clip_on=True,
            )
            labelled_x.append(lam_obs)


def extract_redshift(hdr: fits.Header) -> float | None:
    """
    Return the redshift to use for line annotation.

    Priority order:
      1. Z_OVERRIDE (from parameters cell) — if not None, always wins.
      2. FITS header keywords: REDSHIFT → Z_SPEC → Z_PHOT → ZNEW → Z
      3. None — annotation skipped for this spectrum.
    """
    # 1. Hard-coded override from parameters cell
    try:
        if Z_OVERRIDE is not None:
            z = float(Z_OVERRIDE)
            if 0.0 <= z < 25.0:
                return z
    except (NameError, TypeError, ValueError):
        pass   # Z_OVERRIDE not defined yet or invalid — fall through

    # 2. Per-file header
    for kw in ("REDSHIFT", "Z_SPEC", "Z_PHOT", "ZNEW", "Z"):
        val = hdr.get(kw, None)
        if val is not None:
            try:
                z = float(val)
                if 0.0 <= z < 25.0:
                    return z
            except (TypeError, ValueError):
                pass
    return None


print(f"Emission line catalogue loaded ✓")
print(f"  Groups : {', '.join(EMISSION_LINES.keys())}")
total = sum(len(g['lines']) for g in EMISSION_LINES.values())
print(f"  Total lines defined : {total}")


## 3 · File discovery & inventory

In [ ]:
def find_spec_files(root: Path) -> list[Path]:
    """Recursively find all per-source NIRSpec _spec.fits files."""
    pattern   = str(root / "**" / "*nirspec*spec.fits")
    all_files = sorted(glob.glob(pattern, recursive=True))
    return [Path(f) for f in all_files if "spec-template" not in f]


def classify_files(files: list[Path]) -> dict[str, list[Path]]:
    """Group files by grating/filter optical element token in the filename."""
    groups: dict[str, list[Path]] = {}
    for f in files:
        parts   = f.stem.replace("-", "_").split("_")
        optelem = next(
            (p for p in parts
             if any(g in p for g in ["g140h", "g235h", "g395h", "prism"])),
            "unknown",
        )
        groups.setdefault(optelem, []).append(f)
    return groups


print(f"Searching under: {_MAST_ROOT.resolve()}")
spec_files = find_spec_files(_MAST_ROOT)
groups     = classify_files(spec_files)

if not spec_files:
    display(Markdown(
        "⚠️ **No spectral FITS files found.**  "
        "Check `MAST_ROOT` and run the download script first."
    ))
else:
    rows = ""
    for key in sorted(groups):
        n   = len(groups[key])
        pct = 100 * n / len(spec_files)
        rows += f"<tr><td><code>{key}</code></td><td>{n}</td><td>{pct:.1f}%</td></tr>"

    display(Markdown(f"**Root:** `{_MAST_ROOT.resolve()}`"))
    display(Markdown(
        f"<table>"
        f"<thead><tr><th>Grating / Filter</th><th>Files</th><th>% of total</th></tr></thead>"
        f"<tbody>{rows}</tbody>"
        f"<tfoot><tr><td><strong>TOTAL</strong></td>"
        f"<td><strong>{len(spec_files)}</strong></td><td>100%</td></tr></tfoot>"
        f"</table>"
    ))


## 4 · Metadata & spectrum loading helpers

In [ ]:
_META_KEYS = [
    ("OBJECT",   "Object",    "{}"),
    ("TARGNAME", "Target",    "{}"),
    ("RA_TARG",  "RA",        "{:.5f}°"),
    ("DEC_TARG", "Dec",       "{:.5f}°"),
    ("SRCRA",    "Src RA",    "{:.5f}°"),
    ("SRCDEC",   "Src Dec",   "{:.5f}°"),
    ("REDSHIFT", "z",         "{:.4f}"),
    ("Z_SPEC",   "z_spec",    "{:.4f}"),
    ("Z_PHOT",   "z_phot",    "{:.4f}"),
    ("INSTRUME", "Instr",     "{}"),
    ("GRATING",  "Grating",   "{}"),
    ("FILTER",   "Filter",    "{}"),
    ("DISPERSR", "Disperser", "{}"),
    ("EXPTIME",  "ExpTime",   "{:.0f}s"),
    ("SRCTYPE",  "SrcType",   "{}"),
    ("SLITID",   "SlitID",    "{}"),
    ("SOURCEID", "SourceID",  "{}"),
]


def build_title(hdr: fits.Header, filepath: Path) -> tuple[str, str]:
    tokens = filepath.stem.replace("-", "_").split("_")
    src_id = next((t for t in reversed(tokens) if t.isdigit()), filepath.stem)
    grism  = next(
        (t for t in tokens if any(g in t for g in ["g140h", "g235h", "g395h", "prism"])),
        "?",
    )
    meta = {}
    for kw, label, fmt in _META_KEYS:
        val = hdr.get(kw, None)
        if val is not None and str(val).strip() not in ("", "N/A", "UNKNOWN"):
            try:
                meta[label] = fmt.format(val)
            except (ValueError, TypeError):
                meta[label] = str(val).strip()

    obj_str   = meta.get("Object") or meta.get("Target") or f"Source {src_id}"
    z_str     = meta.get("z") or meta.get("z_spec") or meta.get("z_phot") or "—"
    instr_str = meta.get("Instr", "NIRSpec")
    grat_str  = meta.get("Grating") or meta.get("Disperser") or grism.upper()
    filt_str  = meta.get("Filter", "")
    exp_str   = meta.get("ExpTime", "")
    ra_str    = meta.get("Src RA") or meta.get("RA", "")
    dec_str   = meta.get("Src Dec") or meta.get("Dec", "")

    suptitle = (
        f"{obj_str}  |  {instr_str} / {grat_str}"
        + (f" + {filt_str}" if filt_str else "")
        + f"  |  z = {z_str}"
    )
    parts = []
    if ra_str and dec_str:
        parts.append(f"(RA, Dec) = ({ra_str}, {dec_str})")
    if exp_str:
        parts.append(f"Exp = {exp_str}")
    parts.append(f"File: {filepath.name}")
    return suptitle, "   ·   ".join(parts)


def _find_col(table, name_variants):
    cols = {c.upper() for c in table.names}
    for v in name_variants:
        if v.upper() in cols:
            return table[v]
    return None


def load_1d(hdul):
    for hdu in hdul[1:]:
        if not isinstance(hdu, fits.BinTableHDU):
            continue
        wave = _find_col(hdu.data, _WAVE_NAMES)
        flux = _find_col(hdu.data, _FLUX_NAMES)
        if wave is None or flux is None:
            continue
        err = _find_col(hdu.data, _ERR_NAMES)
        w   = np.asarray(wave, dtype=float).ravel()
        f   = np.asarray(flux, dtype=float).ravel()
        e   = np.asarray(err,  dtype=float).ravel() if err is not None else None
        if np.nanmedian(w) < 50:
            w = w * 1e4
        return w, f, e
    return None


def load_2d(hdul):
    for hdu in hdul[1:]:
        if isinstance(hdu, (fits.ImageHDU, fits.PrimaryHDU)):
            d = hdu.data
            if d is not None and d.ndim == 2 and max(d.shape) > 10:
                return d.astype(float)
    return None


def _error_panel(ax, fpath, msg, idx):
    ax.set_facecolor("#1a0d0d")
    ax.text(0.5, 0.5, f"[{idx}] {fpath.name}\n{msg}",
            transform=ax.transAxes, ha="center", va="center",
            color="#ff6666", fontsize=8, wrap=True)
    ax.set_xticks([])
    ax.set_yticks([])


print("Helpers defined ✓")


## 5 · Plot spectra

**`range` mode** — plots `IDX_START` → `IDX_END` with one panel per spectrum.  
**`z_quad` mode** — plots one 2×2 figure per spectrum, each quadrant showing the
same flux with a different redshift applied to the emission line annotations.
The quadrant where annotated lines align with real flux peaks reveals the
true redshift.

Re-run just this cell after changing any parameter in Cell 1.


In [ ]:
# ---------------------------------------------------------------------------
#  Shared legend helper
# ---------------------------------------------------------------------------

def _make_line_legend(fig, active_groups: set) -> None:
    """Add a compact colour-coded legend for the emission line groups."""
    handles = []
    for grp_key in ["hydrogen", "forbidden", "agn_uv", "agn_coronal", "sfr"]:
        if grp_key not in active_groups:
            continue
        grp = EMISSION_LINES[grp_key]
        handles.append(
            plt.Line2D([0], [0],
                       color=grp["color"], linewidth=1.2, linestyle="--",
                       label=grp_key.replace("_", " ").title())
        )
    if handles:
        fig.legend(
            handles=handles,
            loc="lower center",
            ncol=len(handles),
            fontsize=7,
            framealpha=0.25,
            facecolor="#1a1a2e",
            edgecolor="#444466",
            labelcolor="white",
            bbox_to_anchor=(0.5, 0.0),
        )


# ---------------------------------------------------------------------------
#  Helper: draw one 1-D spectrum panel
# ---------------------------------------------------------------------------

def _draw_1d_panel(
    ax,
    w: np.ndarray,
    f_arr: np.ndarray,
    e,                        # ndarray | None
    redshift: float | None,
    z_label: str,
    global_idx: int,
    fpath,
    suptitle: str,
    subtitle: str,
    col: int,
    active_groups: set,
    show_z_source: str = "",  # extra label suffix, e.g. " (override)"
) -> None:
    """Render a single 1-D spectrum with optional emission line annotation."""
    ax.set_facecolor("#0d0d1a")
    for spine in ax.spines.values():
        spine.set_edgecolor("#2a2a4a")

    fmed = np.nanmedian(f_arr)
    fsig = np.nanstd(f_arr)
    ylo  = fmed - 2.0 * fsig
    yhi  = fmed + 5.0 * fsig

    if e is not None:
        ax.fill_between(w, f_arr - e, f_arr + e,
                        color="#4466aa", alpha=0.35, linewidth=0)

    ax.plot(w, f_arr, color="#88ccff", linewidth=0.7, alpha=0.9)
    ax.axhline(0, color="#555577", linewidth=0.6, linestyle="--")
    ax.set_xlim(w.min(), w.max())
    ax.set_ylim(ylo, yhi)

    if active_groups:
        annotate_emission_lines(
            ax=ax, wave_obs=w, ylo=ylo, yhi=yhi,
            redshift=redshift, groups=active_groups,
        )

    # z label — top-right corner, colour signals whether lines were drawn
    if redshift is not None:
        ax.text(0.98, 0.96, f"z = {redshift:.4f}{show_z_source}",
                transform=ax.transAxes, ha="right", va="top",
                color="#ffcc88", fontsize=7, fontweight="bold", zorder=6)
    else:
        ax.text(0.98, 0.96, "z unknown",
                transform=ax.transAxes, ha="right", va="top",
                color="#886644", fontsize=6.5, style="italic", zorder=6)

    if col == 0:
        ax.set_ylabel("Flux (μJy)", color="#aaaacc", fontsize=8)
    ax.set_xlabel("Wavelength (Å)", color="#aaaacc", fontsize=8)
    ax.tick_params(colors="#aaaacc", labelsize=7)

    # Panel title — in z_quad mode we show the window label; in range mode full metadata
    if z_label:
        ax.set_title(z_label, color="#ffcc88", fontsize=8,
                     fontweight="bold", pad=3, loc="left")
    else:
        ax.set_title(
            f"[{global_idx}]  {suptitle}\n"
            f"$\it{{{subtitle.replace(chr(95), chr(95))}}}$",
            color="white", fontsize=7.5, pad=4, loc="left",
        )


# ---------------------------------------------------------------------------
#  Mode A: range plot  (original one-panel-per-spectrum layout)
# ---------------------------------------------------------------------------

def plot_range(
    files, idx_start, idx_end,
    show_2d=False, show_lines=True, line_groups=None, save=True,
):
    """One panel per spectrum across IDX_START–IDX_END."""
    idx_start = max(0, idx_start)
    idx_end   = min(idx_end, len(files) - 1)
    if idx_start > idx_end:
        display(Markdown(f"⚠️ Nothing to plot: IDX_START ({idx_start}) > IDX_END ({idx_end})."))
        return

    active_groups = (line_groups or set()) if show_lines else set()
    subset = files[idx_start : idx_end + 1]
    display(Markdown(
        f"**[range] Plotting {idx_start}–{idx_end}** "
        f"({len(subset)} panels · lines: "
        f"{', '.join(sorted(active_groups)) if active_groups else 'off'})"
    ))

    n_rows     = (len(subset) + COLS_PER_ROW - 1) // COLS_PER_ROW
    row_h      = PANEL_HEIGHT * (2 if show_2d else 1)
    bottom_pad = 0.06 if active_groups else 0.04
    fig_h      = row_h * n_rows + 1.2

    fig = plt.figure(figsize=(FIG_WIDTH, fig_h), facecolor="#0d0d1a")
    fig.suptitle(
        f"GLASS-JWST NIRSpec Spectra — Abell 2744\n"
        f"Indices {idx_start}–{idx_end}  of  {len(files)} spectra",
        color="white", fontsize=13, fontweight="bold", y=0.998,
    )
    outer = gridspec.GridSpec(
        n_rows, COLS_PER_ROW, figure=fig,
        hspace=0.62, wspace=0.32,
        top=0.96, bottom=bottom_pad, left=0.06, right=0.97,
    )

    for panel_idx, fpath in enumerate(subset):
        row        = panel_idx // COLS_PER_ROW
        col        = panel_idx %  COLS_PER_ROW
        global_idx = idx_start + panel_idx

        if show_2d:
            inner = gridspec.GridSpecFromSubplotSpec(
                2, 1, subplot_spec=outer[row, col],
                height_ratios=[1, 2.5], hspace=0.08,
            )
            ax2d = fig.add_subplot(inner[0])
            ax1d = fig.add_subplot(inner[1])
        else:
            ax1d = fig.add_subplot(outer[row, col])
            ax2d = None

        try:
            with fits.open(fpath, memmap=False) as hdul:
                primary_hdr = hdul[0].header
                combined    = primary_hdr.copy()
                if len(hdul) > 1 and hasattr(hdul[1], "header"):
                    for card in hdul[1].header.cards:
                        if card.keyword not in combined:
                            combined.append(card)
                result_1d = load_1d(hdul)
                img_2d    = load_2d(hdul) if show_2d else None
        except Exception as exc:
            _error_panel(ax1d, fpath, str(exc), global_idx)
            if ax2d:
                ax2d.set_visible(False)
            continue

        suptitle, subtitle = build_title(combined, fpath)
        redshift = extract_redshift(combined)

        if ax2d is not None and img_2d is not None:
            vmed = np.nanmedian(img_2d); vsig = np.nanstd(img_2d)
            ax2d.imshow(img_2d, origin="lower", aspect="auto", cmap="inferno",
                        norm=AsinhNorm(linear_width=max(vsig*0.5,1e-30),
                                       vmin=vmed-vsig, vmax=vmed+8*vsig),
                        interpolation="nearest")
            ax2d.set_xticks([])
            ax2d.set_ylabel("Spatial", color="#aaaacc", fontsize=7)
            ax2d.tick_params(colors="#aaaacc", labelsize=6)
            for spine in ax2d.spines.values():
                spine.set_edgecolor("#333355")
            ax2d.set_facecolor("#0d0d1a")
        elif ax2d is not None:
            ax2d.set_visible(False)

        if result_1d is None:
            _error_panel(ax1d, fpath, "No 1-D spectrum found", global_idx)
            continue

        wave, flux, err = result_1d
        good = np.isfinite(wave) & np.isfinite(flux)
        if err is not None:
            good &= np.isfinite(err) & (err > 0)
        if good.sum() < 3:
            _error_panel(ax1d, fpath, "Insufficient finite pixels", global_idx)
            continue

        w, f_arr = wave[good], flux[good]
        e        = err[good] if err is not None else None

        z_src = ""
        try:
            if Z_OVERRIDE is not None and redshift is not None:
                if abs(float(Z_OVERRIDE) - redshift) < 1e-6:
                    z_src = " (override)"
        except (NameError, TypeError):
            pass

        _draw_1d_panel(
            ax=ax1d, w=w, f_arr=f_arr, e=e,
            redshift=redshift, z_label="",
            global_idx=global_idx, fpath=fpath,
            suptitle=suptitle, subtitle=subtitle,
            col=col, active_groups=active_groups,
            show_z_source=z_src,
        )

    if active_groups:
        _make_line_legend(fig, active_groups)
    plt.tight_layout(rect=[0, bottom_pad, 1, 0.97])

    if save:
        _OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        out = _OUTPUT_DIR / f"glass_jwst_spectra_{idx_start:04d}_{idx_end:04d}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight",
                    facecolor="#0d0d1a", pad_inches=0.08)
        display(Markdown(f"💾 Saved → `{out.resolve()}`"))
    plt.show()


# ---------------------------------------------------------------------------
#  Mode B: z_quad plot  (2×2 per target, four redshift windows)
# ---------------------------------------------------------------------------

def plot_z_quad(
    files, idx_start, idx_end,
    z_windows, show_lines=True, line_groups=None, save=True,
):
    """
    For each spectrum in idx_start–idx_end, produce one figure with a 2×2
    grid of panels — same flux data, four different redshift annotations.
    Each quadrant applies a different z from z_windows so you can visually
    identify which redshift makes the annotated lines land on flux peaks.
    """
    idx_start = max(0, idx_start)
    idx_end   = min(idx_end, len(files) - 1)
    if idx_start > idx_end:
        display(Markdown(f"⚠️ Nothing to plot."))
        return

    active_groups = (line_groups or set()) if show_lines else set()
    n_targets = idx_end - idx_start + 1
    display(Markdown(
        f"**[z\_quad] Plotting {n_targets} target(s), indices {idx_start}–{idx_end}** — "
        f"one 2×2 figure per target · "
        f"z-windows: {[lbl for lbl,_ in z_windows]} · "
        f"lines: {', '.join(sorted(active_groups)) if active_groups else 'off'}"
    ))

    for spec_idx in range(idx_start, idx_end + 1):
        fpath      = files[spec_idx]
        global_idx = spec_idx

        # Load spectrum once
        try:
            with fits.open(fpath, memmap=False) as hdul:
                primary_hdr = hdul[0].header
                combined    = primary_hdr.copy()
                if len(hdul) > 1 and hasattr(hdul[1], "header"):
                    for card in hdul[1].header.cards:
                        if card.keyword not in combined:
                            combined.append(card)
                result_1d = load_1d(hdul)
        except Exception as exc:
            display(Markdown(f"⚠️ [{global_idx}] Could not load `{fpath.name}`: {exc}"))
            continue

        if result_1d is None:
            display(Markdown(f"⚠️ [{global_idx}] No 1-D spectrum in `{fpath.name}` — skipped."))
            continue

        wave, flux, err = result_1d
        good = np.isfinite(wave) & np.isfinite(flux)
        if err is not None:
            good &= np.isfinite(err) & (err > 0)
        if good.sum() < 3:
            display(Markdown(f"⚠️ [{global_idx}] Too few finite pixels — skipped."))
            continue

        w, f_arr = wave[good], flux[good]
        e        = err[good] if err is not None else None

        suptitle, subtitle = build_title(combined, fpath)
        # Header redshift (informational only in this mode)
        hdr_z = extract_redshift_from_header(combined)

        # ── Build figure ──────────────────────────────────────────────────
        fig, axes = plt.subplots(
            2, 2,
            figsize=(FIG_WIDTH, PANEL_HEIGHT * 2.1),
            facecolor="#0d0d1a",
        )
        fig.patch.set_facecolor("#0d0d1a")

        # Super-title: source metadata
        z_hdr_str = f"  |  header z = {hdr_z:.4f}" if hdr_z is not None else ""
        fig.suptitle(
            f"[{global_idx}]  {suptitle}{z_hdr_str}\n"
            f"$\it{{{subtitle.replace(chr(95), chr(95))}}}$\n"
            f"Four redshift windows — annotated lines shift with z",
            color="white", fontsize=9, fontweight="bold",
            y=0.995, va="top",
        )

        for quad_idx, (z_label, z_val) in enumerate(z_windows):
            ax  = axes[quad_idx // 2][quad_idx % 2]
            col = quad_idx % 2

            _draw_1d_panel(
                ax=ax, w=w, f_arr=f_arr, e=e,
                redshift=z_val,
                z_label=z_label,
                global_idx=global_idx, fpath=fpath,
                suptitle=suptitle, subtitle=subtitle,
                col=col, active_groups=active_groups,
                show_z_source="",
            )

        if active_groups:
            _make_line_legend(fig, active_groups)

        plt.tight_layout(rect=[0, 0.06 if active_groups else 0.02, 1, 0.93])

        if save:
            _OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            out = _OUTPUT_DIR / f"glass_jwst_zquad_{global_idx:04d}.png"
            fig.savefig(out, dpi=150, bbox_inches="tight",
                        facecolor="#0d0d1a", pad_inches=0.08)
            display(Markdown(f"💾 [{global_idx}] Saved → `{out.resolve()}`"))
        plt.show()


def extract_redshift_from_header(hdr) -> float | None:
    """Header-only redshift (no Z_OVERRIDE), used for informational display in z_quad."""
    for kw in ("REDSHIFT", "Z_SPEC", "Z_PHOT", "ZNEW", "Z"):
        val = hdr.get(kw, None)
        if val is not None:
            try:
                z = float(val)
                if 0.0 <= z < 25.0:
                    return z
            except (TypeError, ValueError):
                pass
    return None


# ---------------------------------------------------------------------------
#  Dispatcher
# ---------------------------------------------------------------------------

if spec_files:
    if IDX_START > IDX_END:
        IDX_START, IDX_END = IDX_END, IDX_START

    _end_clamped = min(IDX_END, len(spec_files) - 1)
    if _end_clamped < IDX_END:
        display(Markdown(
            f"ℹ️ `IDX_END` clamped {IDX_END} → {_end_clamped} "
            f"(only {len(spec_files)} files available)."
        ))

    if PLOT_MODE == "z_quad":
        plot_z_quad(
            files       = spec_files,
            idx_start   = IDX_START,
            idx_end     = _end_clamped,
            z_windows   = Z_WINDOWS,
            show_lines  = SHOW_LINES,
            line_groups = ANNOTATE_GROUPS,
            save        = SAVE_PNG,
        )
    else:
        plot_range(
            files       = spec_files,
            idx_start   = IDX_START,
            idx_end     = _end_clamped,
            show_2d     = SHOW_2D,
            show_lines  = SHOW_LINES,
            line_groups = ANNOTATE_GROUPS,
            save        = SAVE_PNG,
        )
else:
    display(Markdown("⚠️ No spectra loaded — check `MAST_ROOT` and re-run Cell 3."))
